In [48]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from rdkit import Chem
from rdkit.Chem import AllChem
from mordred import Calculator 


import h5py

# SMILES Representation Methods #


## Mordred ##

In [36]:
ls

blind_test_dataset_preprocessing.ipynb  temp.ipynb


In [37]:
DATA_PATH = Path('/home/gavrilev/Thesis/src/datasets/RSIM/All_sf_dataset_v1.csv')

In [38]:
full_df = pd.read_csv(DATA_PATH, sep='\t')

In [39]:
full_df

,Entry_ID,SMILES,Target_RNA_sequence,Molecule_name,Molecule_ID,Target_RNA_name,Target_RNA_ID,pKd
0,2091,N[C@@H]1C[C@H](N)[C@H]([C@H]([C@H]1OCCCc1nnn(c...,CGCGCAGUCAGUAGCGCG,B-13,Target_lig_1138,RNA hairpin loop IV,Target_381,6.387216
1,162,C1CN(CCC1C(=O)NCCN2CCOCC2)CC3=CC=C(C=C3)Cl,GAGCGUCACACCUUCGGGUGAAGUCGCUC,1-(4-chlorobenzyl)-N-(2-morpholinoethyl)piperi...,Target_lig_132,A-site 16S rRNA_E_coli (1),Target_3,3.602060
2,270,c1c(ccc2Sc3ccccc3N(c12)CCCNCN(CO)CO)C(=O)C,GGCAGAUCUGAGCCUGGGAGCUCUCUGCC,3_phenothiazine_analog_1,Target_lig_210,HIV-1 TAR RNA,Target_19,2.522879
3,1327,C(CN1C=C(CNC2=C3C=CC=CC3=[NH+]C3=C2OC2=C3C=CC=...,GGGUUAGGGU,Quindoline CK1-14,Target_lig_723,TRF2:TERRA G-quadruplex (TERRA22):Telomeric du...,Target_262,6.272784
4,1907,C1=CC=C(C=C1)C2=NN=C(O2)C3=NC4=CC=CC=C4C(=N3)N...,UGUGGGAGGGGCGGGUCUGGG,NRAS SAR 7,Target_lig_1057,NRAS G4 motif,Target_259,6.443697
...,...,...,...,...,...,...,...,...
1434,2119,CN(C)C1=CC2=C(C=C1)C(=C3C=CC(=[N+](C)C)C=C3O2)...,GGUACCCGUCUGGCGAGAGCCAGGUAACGAAUGGUACC,Tetramethylrosamine (TMR),Target_lig_1151,Aptamer mutant U9,Target_388,6.853872
1435,1248,C1=CC(=CC=C1C2=CC=C(O2)C3=CC=C(C=C3)C(=[NH2+])...,GGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCA...,DB75,Target_lig_697,"HTT exon1 RNA, 18 repeats",Target_215,4.222573
1436,1336,OC(Cn1c2ccc(cc2c2c1ccc(c2)Br)Br)CNc1ccc(cc1)N1...,GGGUUGACUGUUGAAUCUCAUGGCAACCC,Aniline_analog_7,Target_lig_731,miR-21 hairpin,Target_265,5.677781
1437,1810,NCCCC(=O)NCCCN1Cc2ccc3c(n2)nc(cc3)NC(=O)OCC/C=...,GCUGAUUUCUUUUGGUGUUCAGAGUGGUCGGUGGUUGGUGGUGGGU...,CMBL-3aL,Target_lig_1018,R-Seq2,Target_322,5.882729


In [40]:
unique_smi = full_df['SMILES'].unique()
len(unique_smi)

759

In [41]:
calc = Calculator(descriptors, ignore_3D=True)
len(calc.descriptors)

1613

In [43]:
nan_mols = [smi for smi in unique_smi if Chem.MolFromSmiles(smi) is None]

[18:16:58] non-ring atom 1 marked aromatic
[18:16:58] Explicit valence for atom # 7 N, 4, is greater than permitted
[18:16:58] non-ring atom 1 marked aromatic
[18:16:58] non-ring atom 1 marked aromatic
[18:16:58] Explicit valence for atom # 4 N, 4, is greater than permitted
[18:16:58] Explicit valence for atom # 8 N, 4, is greater than permitted
[18:16:58] Explicit valence for atom # 16 N, 4, is greater than permitted
[18:16:58] Explicit valence for atom # 16 N, 4, is greater than permitted
[18:16:58] non-ring atom 1 marked aromatic
[18:16:58] Explicit valence for atom # 16 N, 4, is greater than permitted


In [44]:
nan_mols_count_rows = 0
for n_mol in nan_mols:
    nan_mols_count_rows += full_df[full_df['SMILES'] == n_mol].shape[0]
len(nan_mols), nan_mols_count_rows,  round(nan_mols_count_rows / full_df.shape[0], 5) * 100

(10, 15, 1.042)

In [45]:
def preprocess_mordred_descriptors(mordred_ls, smi_ls):
    desc_df = pd.DataFrame(mordred_ls)
    desc_df.insert(0, "SMILES", list(smi_ls))

    # numeric stability: coerce non-numeric (e.g., Missing) -> NaN, then fill
    desc_df.iloc[:, 1:] = (
        desc_df.iloc[:, 1:]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype(np.float32)
    )

    # drop columns with constant values (only descriptors, keep SMILES)
    desc_cols = desc_df.columns[1:]
    keep_nonconst = desc_df[desc_cols].nunique() > 1
    desc_df = pd.concat([desc_df[["SMILES"]], desc_df.loc[:, desc_cols[keep_nonconst]]], axis=1)

    # variance threshold (only descriptors)
    selector = VarianceThreshold(threshold=1e-2)
    reduced_descriptors = selector.fit_transform(desc_df.iloc[:, 1:])
    keep_cols = desc_df.iloc[:, 1:].columns[selector.get_support()]

    desc_df = pd.DataFrame(reduced_descriptors, columns=keep_cols)
    desc_df.insert(0, "SMILES", list(smi_ls))

    return desc_df

In [46]:
def calc_mordred_desc(smi_ls, ignore_3D=True):
    calculator = Calculator(descriptors, ignore_3D=ignore_3D)
    print(len(calculator.descriptors))

    mols = [Chem.MolFromSmiles(smi) for smi in smi_ls]
    descriptors_ls = []

    desc_names = [str(d) for d in calculator.descriptors]

    for mol in mols:
        if mol is not None:
            result = calculator(mol)
            desc_dict = {str(k): v for k, v in result.asdict().items()}
        else:
            desc_dict = {name: 0.0 for name in desc_names}

        descriptors_ls.append(desc_dict)

    desc_df = preprocess_mordred_descriptors(descriptors_ls, smi_ls)

    desc_map = {
        smi: desc_df.loc[i, desc_df.columns[1:]].values.astype(np.float32)
        for i, smi in enumerate(smi_ls)
    }

    return desc_df, desc_map

In [49]:
feature_df, feature_dict = calc_mordred_desc(unique_smi)

1613


[18:28:59] non-ring atom 1 marked aromatic
[18:28:59] Explicit valence for atom # 7 N, 4, is greater than permitted
[18:28:59] non-ring atom 1 marked aromatic
[18:28:59] non-ring atom 1 marked aromatic
[18:28:59] Explicit valence for atom # 4 N, 4, is greater than permitted
[18:28:59] Explicit valence for atom # 8 N, 4, is greater than permitted
[18:28:59] Explicit valence for atom # 16 N, 4, is greater than permitted
[18:28:59] Explicit valence for atom # 16 N, 4, is greater than permitted
[18:28:59] non-ring atom 1 marked aromatic
[18:28:59] Explicit valence for atom # 16 N, 4, is greater than permitted
/home/gavrilev/.conda/envs/chemprop/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


In [200]:
def scale(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standard-scale all numeric feature columns (assumes first column is SMILES/ID),
    then return a DataFrame with the original first column + scaled features.
    """
    if df.shape[0] <= 2:
        return df

    smiles_col = df.iloc[:, 0].copy()
    X = df.iloc[:, 1:].copy()

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=df.index)
    scaled_df.insert(0, df.columns[0], smiles_col)

    return scaled_df    


In [202]:
scaled_df = scale(feature_df)

In [204]:
scaled_df.shape

(759, 1206)

In [196]:
feature_dict['Ccc1nc2ccccc2cc1C(=O)O']

array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)

In [31]:
def generate_morgan_counts(smiles_list, radius=2, nbits=1028):
    fp_mat = np.zeros((len(smiles_list), nbits), dtype=np.float32)

    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        # in case molecule is None, we'll assign 0 vector where |vector| = nbits
        if mol is None:
            continue

        fp = AllChem.GetHashedMorganFingerprint(mol, radius, nBits=nbits)
        for idx, val in fp.GetNonzeroElements().items():
            fp_mat[i, idx] = val

    cols = [f"MCFP_{j}" for j in range(nbits)]
    df = pd.DataFrame(fp_mat, columns=cols)
    df.insert(0, "SMILES", smiles_list)

    return df

In [32]:
morgan_df = generate_morgan_counts(unique_smi)

[17:26:04] non-ring atom 1 marked aromatic
[17:26:04] Explicit valence for atom # 7 N, 4, is greater than permitted
[17:26:04] non-ring atom 1 marked aromatic
[17:26:04] non-ring atom 1 marked aromatic
[17:26:05] Explicit valence for atom # 4 N, 4, is greater than permitted
[17:26:05] Explicit valence for atom # 8 N, 4, is greater than permitted
[17:26:05] Explicit valence for atom # 16 N, 4, is greater than permitted
[17:26:05] Explicit valence for atom # 16 N, 4, is greater than permitted
[17:26:05] non-ring atom 1 marked aromatic
[17:26:05] Explicit valence for atom # 16 N, 4, is greater than permitted


In [30]:
morgan_df['MCFP_2021'].unique()

array([0., 1., 2., 3.], dtype=float32)